## 1. Imports & Load Raw Data

In [32]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os.path as path

In [33]:
df = pd.read_csv('dataset/diamonds.csv')
y_target = 'price'
df.head(5)

,carat,cut,color,clarity,depth,table,price,x,y,z
0,0.23,Ideal,E,SI2,61.5,55.0,326,3.95,3.98,2.43
1,0.21,Premium,E,SI1,59.8,61.0,326,3.89,3.84,2.31
2,0.23,Good,E,VS1,56.9,65.0,327,4.05,4.07,2.31
3,0.29,Premium,I,VS2,62.4,58.0,334,4.20,4.23,2.63
4,0.31,Good,J,SI2,63.3,58.0,335,4.34,4.35,2.75


## 2. Handle Missing / Invalid Values

In [34]:
missing_feature = df.columns[df.isnull().any()].to_list()
print(f'Missing values: {df.isnull().sum().sum()}')
print(f'Duplicates: {df.duplicated().sum()}')
print(f'Missing Feature:\n{missing_feature}')

Missing values: 0
Duplicates: 146
Missing Feature:
[]


In [35]:
for col in missing_feature:
    missing_percentage = (df[col].isnull().sum() / len(df)) * 100
    if missing_percentage > 5.0 :
        df = df.drop(columns=col)
    else:
        if df[col].dtype == 'object':
            df[col] = df[col].fillna(df[col].mode()[0])
        else:
            skewness = df[col].skew()
            if abs(skewness) < 0.5:
                df[col] = df[col].fillna(df[col].mean()).round(2)
            else:
                df[col] = df[col].fillna(df[col].median()).round(2)

print(f'Missing values: {df.isnull().sum().sum()}')
print(f'Missing Feature:\n{missing_feature}')

Missing values: 0
Missing Feature:
[]


## 3. Handling Duplicated

In [36]:
df = df.drop_duplicates(keep='last')

jumlah_duplikat = df.duplicated().sum()
print(f"Jumlah data duplikat: {jumlah_duplikat}")

Jumlah data duplikat: 0


## 4. Analisis & Handling Outliers

In [37]:
feature_numerik = df.select_dtypes(include=np.number).columns
Q1 = df[feature_numerik].quantile(0.25)
Q3 = df[feature_numerik].quantile(0.75)
IQR = Q3-Q1

lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

outliers = df.loc[((df[feature_numerik] < lower_bound) | (df[feature_numerik] > upper_bound)).any(axis=1)]
print(f"Jumlah Outlier Sebelum Dihapus: {outliers.shape[0]}")

#delete outliers
df = df.loc[((df[feature_numerik] >= lower_bound) & (df[feature_numerik] <= upper_bound)).all(axis=1)]
outliers = df.loc[((df[feature_numerik] < lower_bound) | (df[feature_numerik] > upper_bound)).any(axis=1)]
print(f"Jumlah Outlier Sesudah Dihapus: {outliers.shape[0]}")


Jumlah Outlier Sebelum Dihapus: 6378
Jumlah Outlier Sesudah Dihapus: 0


## 5. Feature Engineering

In [38]:
df['Volume_diamond'] = df['x'] * df['y'] * df['z']
df['Ratio'] = np.where(df['y'] == 0, 0, df['x'] / df['y'])
df['surface_area'] = 2 *((df['x'] * df['y']) + (df['y'] * df['z']) + (df['x'] * df['z']))

count_xy = (df['x'] + df['y']) / 2
df['Calculated_depth'] = np.where(count_xy == 0, 1, (df['z'] / count_xy) * 100)
df['depth_deviation'] = np.abs(df['depth'] - df['Calculated_depth'])

df['carat_per_volume'] = np.where(df['Volume_diamond'] == 0,1,df['carat'] / df['Volume_diamond'])

In [39]:
cut_mapping = { 'Ideal': 5,  'Premium': 4,  'Very Good': 3,  'Good': 2,  'Fair': 1}
color_mapping = { 'D': 7, 'E': 6, 'F': 5, 'G': 4, 'H': 3, 'I': 2, 'J': 1}
clarity_mapping = { 'IF': 8, 'VVS1': 7, 'VVS2': 6, 'VS1': 5, 'VS2': 4, 'SI1': 3, 'SI2': 2, 'I1': 1}

df['cut_score'] = df['cut'].apply(lambda x : cut_mapping.get(x,0))
df['color_score'] = df['color'].apply(lambda x : color_mapping.get(x,0))
df['clarity_score'] = df['clarity'].apply(lambda x : clarity_mapping.get(x,0))

df['qualty_index'] = df['cut_score'] * df['color_score'] * df['clarity_score']

## 6.Save Cleaned Dataset

In [40]:
file_path = 'dataset/diamonds_CLEANING.csv'

if not path.exists(file_path):
    df.to_csv(file_path, index=False)
    print('File belum ada. Berhasil menyimpan dataset baru!')
else:
    print('File sudah ada. Proses penyimpanan CSV dilewati (skip)')

df.head()

File belum ada. Berhasil menyimpan dataset baru!


,carat,cut,color,clarity,depth,table,price,x,y,z,Volume_diamond,Ratio,surface_area,Calculated_depth,depth_deviation,carat_per_volume,cut_score,color_score,clarity_score,qualty_index
0,0.23,Ideal,E,SI2,61.5,55.0,326,3.95,3.98,2.43,38.202030,0.992462,69.9818,61.286255,0.213745,0.006021,5,6,2,60
1,0.21,Premium,E,SI1,59.8,61.0,326,3.89,3.84,2.31,34.505856,1.013021,65.5878,59.767141,0.032859,0.006086,4,6,3,72
3,0.29,Premium,I,VS2,62.4,58.0,334,4.20,4.23,2.63,46.724580,0.992908,79.8738,62.396204,0.003796,0.006207,4,2,4,32
4,0.31,Good,J,SI2,63.3,58.0,335,4.34,4.35,2.75,51.917250,0.997701,85.5530,63.291139,0.008861,0.005971,2,1,2,4
5,0.24,Very Good,J,VVS2,62.8,57.0,336,3.94,3.96,2.48,38.693952,0.994949,70.3888,62.784810,0.015190,0.006203,3,1,6,18
